# Tarea 2
Integrantes: Claudio Alvial, Cristóbal Navarro, Cristopher Vasques.


## Parte 1

## Creación de CNN propia
Primero inst

## Parte 2

### Faster R-CNN: entrenamiento e inferencia

Ajustamos pesos preentrenados a las 28 clases de FjordFish. Configuraci?n inicial: 10 ?pocas, batch de 2, SGD con learning rate 0.005 y parada tras 3 ?pocas sin mejora.

Cada ?poca recorre train y luego val, sin actualizar pesos durante validaci?n. Se muestra p?rdida de entrenamiento y validaci?n; esta p?rdida **no es mAP**. Se guardan `best.pt`, `last.pt` e `historial.csv` en `resultados/faster_entrenamiento`. Al terminar se usa el mejor modelo para detectar sobre val.

Esta celda entrena al ejecutarse y puede tardar bastante. Ajusta las variables al principio. Si se agota la memoria de GPU, baja `BATCH_SIZE` a 1.


In [ ]:
import os
import re
from pathlib import Path

import torch
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2,
    FasterRCNN_ResNet50_FPN_V2_Weights,
)
from torchvision.utils import draw_bounding_boxes
from torchvision.transforms.functional import to_pil_image
from torchvision.io import ImageReadMode, read_image
import pandas as pd

#=== Variables varias ===
CARPETA_ENTRADA = "data/FjordFish/images/val"
CARPETA_SALIDA  = Path("resultados")      
ARCHIVO_CSV     = "detecciones.csv" 
UMBRAL_CONFIANZA    = 0.9         
EXTENSION = ".jpg" 


# Creamos la carpeta de salida de las imagenes procesadas
os.makedirs(CARPETA_SALIDA, exist_ok=True)


#=== Entrenamiento con las clases y etiquetas de FjordFish ===
from entrenamiento import entrenar_faster

EPOCAS = 10
BATCH_SIZE = 2
LEARNING_RATE = 0.005
PACIENCIA = 3
TAMANO_IMAGEN = 640

model, nombres_clases, historial = entrenar_faster(
    raiz="data/FjordFish",
    epocas=EPOCAS,
    batch=BATCH_SIZE,
    lr=LEARNING_RATE,
    paciencia=PACIENCIA,
    tamano=TAMANO_IMAGEN,
)
dispositivo = next(model.parameters()).device
# El umbral alto es solo para las predicciones finales, no para entrenar.
model.roi_heads.score_thresh = UMBRAL_CONFIANZA
pesos = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT

#Obtenemos la transformación de la imagen usada en el entrenamiento
preprocess = pesos.transforms()

#=== Guardamos las rutas de las imagenes ===
carpeta = Path(CARPETA_ENTRADA)
rutas = []

# Guardamos las rutas de las imagenes y luego las ordenamos de menor a mayor por su numero de foto
for archivo in sorted(carpeta.iterdir()):
    if archivo.suffix == EXTENSION:
        rutas.append(archivo)

#Función para obtener el numero de cada foto
def extraer_numero(path):
    numeros = re.findall(r'\d+', path.name)
    if numeros:
        return int(numeros[0])
    else:
        return 0


#Ordenamos las rutas por el numero de foto
rutas.sort(key=extraer_numero)
print(f"Se encontraron {len(rutas)} imágenes en '{CARPETA_ENTRADA}'")



#=== procesamos las imagenes y guardamos su información ===

#En esta lista van guardados los objetos encontrados, junto con su información más relevante
registros = []  

for ruta in rutas:
    print(f"Procesando: {ruta.name}")
    # leemos la imagen
    img = read_image(str(ruta), mode=ImageReadMode.RGB)

    # Preprocesar y mover al dispositivo
    batch = [preprocess(img).to(dispositivo)]

    # Desactivamos el calculo de la gradiente para adivinar y no entrenar
    with torch.no_grad():
        prediction = model(batch)[0]


    cajas  = prediction["boxes"].cpu()
    etiqueta = prediction["labels"].cpu()
    confianza = prediction["scores"].cpu()

    # Caso para cuando no se haya encontrado ningún objeto
    if len(cajas) == 0:
        registros.append({
            "archivo": ruta.name,
            "etiqueta": None,
            "confianza": None,
            "x1": None, "y1": None, "x2": None, "y2": None,
        })
        
    # Caso en donde encontramos objetos en las imagenes
    else:
        # Nombres de las categorías
        nombres = [nombres_clases[i] for i in etiqueta.tolist()]

        # Guardamos en los registros la informacion de cada objeto encontrado
        for nombre, score, box in zip(nombres, confianza.tolist(), cajas.tolist()):
            x1, y1, x2, y2 = box
            registros.append({
                "archivo": ruta.name,
                "etiqueta": nombre,
                "confianza": round(score, 4),
                "x1": round(x1, 1), "y1": round(y1, 1),
                "x2": round(x2, 1), "y2": round(y2, 1),
            })

    # Dibujar las cajas sobre la imagen original
    if len(cajas) > 0:
        nombres = [nombres_clases[i] for i in etiqueta.tolist()]
        img_anotada = draw_bounding_boxes(
            img,
            boxes=cajas,
            labels=nombres,
            colors="red",
            width=4,
            font_size=30,
        )
    else:
        img_anotada = img

    to_pil_image(img_anotada).save(CARPETA_SALIDA / ruta.name)


#=== Creamos el dt y lo exportamos ===
df = pd.DataFrame(
    registros,
    columns=["archivo", "etiqueta", "confianza", "x1", "y1", "x2", "y2"],
)
df.to_csv(ARCHIVO_CSV, index=False, encoding="utf-8")

print("\n===== RESUMEN =====")
print(f"Total de filas (detecciones): {len(df)}")
print(f"Imágenes procesadas: {df['archivo'].nunique()}")
print(f"\nPrimeras filas:\n{df.head(10)}")
print(f"\nConteo por etiqueta:\n{df['etiqueta'].value_counts()}")
print(f"\nResultados guardados en: {ARCHIVO_CSV}")
print(f"Imágenes anotadas en: {CARPETA_SALIDA}/")

## YOLO: entrenamiento e inferencia

YOLOv8n se ajusta a FjordFish durante un m?ximo de 10 ?pocas con AdamW, learning rate 0.001, batch de 2 y paciencia de 3. Ultralytics registra p?rdidas, precisi?n, recall y mAP de validaci?n, y guarda los mejores pesos y gr?ficos dentro de `resultados_yolo/entrenamiento*`.

Se genera una configuraci?n con las rutas locales del dataset. La celda requiere `ultralytics` y puede descargar los pesos iniciales. Las predicciones finales usan `best.pt` sobre val.


In [ ]:
import os
import re
from pathlib import Path
import pandas as pd
from ultralytics import YOLO

#=== Variables Varias ===
CARPETA_ENTRADA = "data/FjordFish/images/val"
CARPETA_SALIDA  = Path("resultados_yolo")
ARCHIVO_CSV     = "detecciones_yolo.csv"
EXTENSION = ".jpg"
UMBRAL_CONFIANZA    = 0.7

os.makedirs(CARPETA_SALIDA, exist_ok=True)

# Entrenamos YOLOv8n con las 28 clases de FjordFish.
import torch
from entrenamiento import preparar_yaml_yolo

EPOCAS = 10
BATCH_SIZE = 2
LEARNING_RATE = 0.001
PACIENCIA = 3
TAMANO_IMAGEN = 640

config_dataset = preparar_yaml_yolo("data/FjordFish", "resultados_yolo")
model = YOLO("yolov8n.pt")
metricas = model.train(
    data=str(config_dataset),
    epochs=EPOCAS,
    batch=BATCH_SIZE,
    imgsz=TAMANO_IMAGEN,
    optimizer="AdamW",
    lr0=LEARNING_RATE,
    patience=PACIENCIA,
    device=0 if torch.cuda.is_available() else "cpu",
    workers=0,  # Compatible con notebooks en Windows.
    seed=42,
    project="resultados_yolo",
    name="entrenamiento",
    save=True,
    plots=True,
    val=True,
)
# Usamos los mejores pesos para las imagenes y el CSV siguientes.
model = YOLO(str(Path(model.trainer.save_dir) / "weights" / "best.pt"))


carpeta = Path(CARPETA_ENTRADA)
rutas = []

#Misma lógica que en el modelo anterior
for archivo in carpeta.iterdir():
    #Obtenemos solo las imagenes de la carpeta, asi evitamos obtener los archivos ocultoas
    if archivo.suffix == EXTENSION:
        rutas.append(archivo)

def extraer_numero(path):
    numeros = re.findall(r'\d+', path.name)
    if numeros:
        return int(numeros[0])
    else:
        return 0

#Ordenamos en base a los numeros
rutas.sort(key=extraer_numero)
print(f"Se encontraron {len(rutas)} imágenes en '{CARPETA_ENTRADA}'")


#=== procesamos las imagenes y guardamos su información ===
registros = []

for ruta in rutas:
    print(f"Procesando: {ruta.name}")

    # generamos las inferencias con verbose false, para no ver tanto log
    results = model.predict(source=str(ruta), conf=UMBRAL_CONFIANZA, verbose=False)[0]

    # YOLO dibuja y guarda la imagen anotada automáticamente en la carpeta de destino
    results.save(filename=str(CARPETA_SALIDA / ruta.name))

    # Extraer las detecciones obtenidas
    boxes = results.boxes

    # Si no hubo detecciones que superen el umbral
    if len(boxes) == 0:
        registros.append({
            "archivo": ruta.name,
            "etiqueta": None,
            "confianza": None,
            "x1": None, "y1": None, "x2": None, "y2": None,
        })
    else:
        print("Se encontro un objeto")
        # Convertir datos de tensores a arrays de Python/NumPy
        xyxy_list = boxes.xyxy.cpu().numpy()     # Coordenadas [x1, y1, x2, y2]
        conf_list = boxes.conf.cpu().numpy()     # Scores de confianza
        cls_list  = boxes.cls.cpu().numpy()      # IDs numéricos de las clases

        for box, conf, cls_id in zip(xyxy_list, conf_list, cls_list):
            x1, y1, x2, y2 = box
            nombre_clase = model.names[int(cls_id)]  # Traducción de ID a Nombre

            registros.append({
                "archivo": ruta.name,
                "etiqueta": nombre_clase,
                "confianza": round(float(conf), 4),
                "x1": round(float(x1), 1),
                "y1": round(float(y1), 1),
                "x2": round(float(x2), 1),
                "y2": round(float(y2), 1),
            })

# =========================================================
# 5. CREAR DATAFRAME Y EXPORTAR
# =========================================================
df = pd.DataFrame(
    registros,
    columns=["archivo", "etiqueta", "confianza", "x1", "y1", "x2", "y2"],
)
df.to_csv(ARCHIVO_CSV, index=False, encoding="utf-8")

print("\n===== RESUMEN YOLO =====")
print(f"Total de filas (detecciones): {len(df)}")
print(f"Imágenes procesadas: {df['archivo'].nunique()}")
print(f"\nPrimeras filas:\n{df.head(10)}")
print(f"\nConteo por etiqueta:\n{df['etiqueta'].value_counts()}")
print(f"\nResultados guardados en: {ARCHIVO_CSV}")
print(f"Imágenes anotadas en: {CARPETA_SALIDA}/")